# Data Cleaning & Analysis with Pandas — Solutions

Based on the slides *Data Cleaning & Analysis with Pandas*. Each exercise is tagged **Usage** (check that you can use a function), **Think** (decide and explain), **Predict** or **Debug**.

| Part | Content |
|---|---|
| Setup | Create `hr_employees.csv` |
| A | Function checks: Series, DataFrame, read/inspect, missing data, duplicates, text, types, filtering, new columns, groupby, merge |
| B | Clean and analyse the HR data, with decisions to justify |
| C | Predict and debug |

**Note:** write your own code in each empty cell, do the exercises in order (later ones reuse earlier variables), and write explanations in the answer cell. 

In [1]:
import csv
import numpy as np
import pandas as pd
from datetime import date, timedelta

# ---- creates hr_employees.csv (synthetic, deliberately messy). Run once; do not edit. ----
head = """emp_id,join_date,name,dept,age,salary,city,status,exit_date
101,2024-01-15,Alice Brown,Admin,28,"$52,000",Delhi,Active,
102,15/02/2024, bob smith ,TECHNICAL,34,61000,delhi_,Active,
103,2024-03-01,Carol Jones,staff,,"48,500",Mumbai,Active,
104,2024-03-05,David Lee,Admin,250,"$75,000",Pune,Active,
104,2024-03-05,David Lee,Admin,250,"$75,000",Pune,Active,
105,2024-03-10,Eva Rao,Technical,29,,CHENNAI,Resigned,2024-06-30
106,"Mar 12, 2024",Farid Khan,Technical,-5,58000,Mumbai_,active,
107,2024-04-02,,Admin,41,N/A,Delhi,Resigned,2024-05-20
"""
rng = np.random.default_rng(5)
first = ["Anil", "Bela", "Chirag", "Divya", "Ehsan", "Fiza", "Gopal", "Hema", "Irfan", "Jaya", "Karan", "Leela"]
last = ["Rao", "Nair", "Singh", "Das", "Khan", "Iyer", "Menon", "Gill", "Verma", "Bose"]

def messy(t, p=0.3):
    r = rng.random()
    if r < p / 3: return " " + t.lower() + " "
    if r < 2 * p / 3: return t.upper()
    if r < p: return t + " "
    return t

rows = []
for emp in range(108, 141):
    jd = date(2024, 1, 1) + timedelta(days=int(rng.integers(0, 366)))
    f = int(rng.integers(0, 3))
    js = jd.strftime("%d/%m/%Y") if (f == 1 and jd.day > 12) else jd.strftime("%b %d, %Y") if f == 2 else jd.isoformat()
    dept = str(rng.choice(["Admin", "Technical", "Staff"], p=[.35, .4, .25]))
    sal = int({"Admin": 55000, "Technical": 65000, "Staff": 45000}[dept] * rng.uniform(.85, 1.25) // 500 * 500)
    salary = [f"${sal:,}", f"{sal:,}", str(sal), "", "TBD"][int(rng.integers(0, 5))]
    age = "" if rng.random() < .1 else str(int(rng.integers(22, 58)))
    city = str(rng.choice(["Delhi", "Mumbai", "Pune", "Chennai", "Kolkata"]))
    r = rng.random(); city = city.lower() if r < .15 else city.upper() if r < .3 else city + "_" if r < .4 else city
    resigned = rng.random() < .25
    status = messy("Resigned" if resigned else "Active", .3)
    ex = (jd + timedelta(days=int(rng.integers(30, 200)))).isoformat() if resigned else ""
    name = "" if emp == 120 else messy(f"{rng.choice(first)} {rng.choice(last)}")
    rows.append([str(emp), js, name, messy(dept, .4), "199" if emp == 130 else age, salary, city, status, ex])
rows += [rows[3], rows[17]]                       # two repeated uploads

with open("hr_employees.csv", "w", newline="") as fh:
    fh.write(head)
    csv.writer(fh).writerows(rows)
print("hr_employees.csv created")

hr_employees.csv created


---
## Part A — Function checks 

Small examples taken from the slides. The expected result is given so you can check yourself. Write your own code in each cell; do not copy it from the slides.

### Exercise 1.1 
Create a **Series** from the list `[85, 90, 78]` with index `Alice`, `Bob`, `Charlie`, and print it.

Expected: values 85, 90, 78 with `dtype: int64`.

In [2]:
scores = pd.Series([85, 90, 78], index=['Alice', 'Bob', 'Charlie'])
print(scores)

Alice      85
Bob        90
Charlie    78
dtype: int64


### Exercise 1.2 
Create the same Series from a **dictionary** `{'Alice': 85, 'Bob': 90, 'Charlie': 78}`.

In [3]:
s_dict = pd.Series({'Alice': 85, 'Bob': 90, 'Charlie': 78})
print(s_dict)

Alice      85
Bob        90
Charlie    78
dtype: int64


### Exercise 1.3 
Create a Series from the **scalar** `10` with index `a`, `b`, `c` and `dtype='float64'`.

Expected: `10.0` three times.

In [4]:
s_scalar = pd.Series(10, index=['a', 'b', 'c'], dtype='float64')
print(s_scalar)

a    10.0
b    10.0
c    10.0
dtype: float64


### Exercise 1.4 
Create a **DataFrame** from a dictionary with `Name` = Alice, Bob and `Age` = 25, 30. Print it.

In [5]:
df = pd.DataFrame({'Name': ['Alice', 'Bob'], 'Age': [25, 30]})
print(df)

    Name  Age
0  Alice   25
1    Bob   30


### Exercise 1.5 
Create a DataFrame from the 2D list `[[9, 3], [10, 4]]` with row labels `R1`, `R2` and column names `Price`, `Qty`.

In [6]:
df_2d = pd.DataFrame([[9, 3], [10, 4]], index=['R1', 'R2'], columns=['Price', 'Qty'])
print(df_2d)

    Price  Qty
R1      9    3
R2     10    4


### Exercise 2.1 
Read `hr_employees.csv` into a variable `raw` with `read_csv` (give `sep` and `header` explicitly) and print its shape.

Expected: `(43, 9)`.

In [7]:
raw = pd.read_csv('hr_employees.csv', sep=',', header=0)
print(raw.shape)

(43, 9)


### Exercise 2.2 
Show the first 3 rows and the last 2 rows of `raw`.

In [8]:
print(raw.head(3))
print(raw.tail(2))

   emp_id   join_date         name       dept   age   salary    city  status  \
0     101  2024-01-15  Alice Brown      Admin  28.0  $52,000   Delhi  Active   
1     102  15/02/2024   bob smith   TECHNICAL  34.0    61000  delhi_  Active   
2     103  2024-03-01  Carol Jones      staff   NaN   48,500  Mumbai  Active   

  exit_date  
0       NaN  
1       NaN  
2       NaN  
    emp_id     join_date         name   dept   age   salary     city   status  \
41     111    2024-07-03     Anil Das  Staff  32.0   43,500  Chennai  Active    
42     125  Mar 13, 2024   irfan rao   ADMIN   NaN  $51,000   Mumbai   ACTIVE   

   exit_date  
41       NaN  
42       NaN  


### Exercise 2.3 
Print `raw.shape` and the column names as a Python list.

In [9]:
print(raw.shape)
print(list(raw.columns))

(43, 9)
['emp_id', 'join_date', 'name', 'dept', 'age', 'salary', 'city', 'status', 'exit_date']


### Exercise 2.4 · Think
Run `raw.info()`. Which columns are text (dtype `object`, shown as `str` in pandas 3), and which of those *should* be a number or a date? Which column has the most missing values?

In [10]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   emp_id     43 non-null     int64  
 1   join_date  43 non-null     object 
 2   name       41 non-null     object 
 3   dept       43 non-null     object 
 4   age        35 non-null     float64
 5   salary     36 non-null     object 
 6   city       43 non-null     object 
 7   status     43 non-null     object 
 8   exit_date  14 non-null     object 
dtypes: float64(1), int64(1), object(7)
memory usage: 3.1+ KB


**Model answer:** `name`, `dept`, `city`, `status` are text (correct). `join_date`, `exit_date` should be dates, and `salary` should be a number, but all three are `object`. `exit_date` has the most missing values (employees who are still active).

### Exercise 2.5 
For `df = pd.DataFrame({'Salary': [50000, 60000, 75000, 120000]})` run (a) `describe(include='all')` and (b) `describe(percentiles=[0.1, 0.9])`.

Expected: mean 76250, median (50%) 67500, std about 30923.29.

In [11]:
df = pd.DataFrame({'Salary': [50000, 60000, 75000, 120000]})
print(df.describe(include='all'))
print(df.describe(percentiles=[0.1, 0.9]))

              Salary
count       4.000000
mean    76250.000000
std     30923.292192
min     50000.000000
25%     57500.000000
50%     67500.000000
75%     86250.000000
max    120000.000000
              Salary
count       4.000000
mean    76250.000000
std     30923.292192
min     50000.000000
10%     53000.000000
50%     67500.000000
90%    106500.000000
max    120000.000000


### Exercise 3.1 
For `df = pd.DataFrame({'A': [1, np.nan, 3], 'B': [np.nan, np.nan, 6]})` print `df.isna()` and the number of missing values per column.

Expected counts: `A` 1, `B` 2.

In [12]:
df = pd.DataFrame({'A': [1, np.nan, 3], 'B': [np.nan, np.nan, 6]})
print(df.isna())
print(df.isna().sum())

       A      B
0  False   True
1   True   True
2  False  False
A    1
B    2
dtype: int64


### Exercise 3.2 · Think
Compare `df['A'] == np.nan` with `df['A'].isna()`. Why do they give different results? Write your explanation in the answer cell.

In [13]:
print(df['A'] == np.nan)
print(df['A'].isna())

0    False
1    False
2    False
Name: A, dtype: bool
0    False
1     True
2    False
Name: A, dtype: bool


**Model answer:** NaN is defined (IEEE 754) so that every comparison with it is False, even `NaN == NaN`. So `== np.nan` is False everywhere and finds nothing. `isna()` tests whether a value is missing, so it finds the NaN.

### Exercise 3.3 
For `df2 = pd.DataFrame({'A': [1, np.nan, 3], 'B': [4, 5, np.nan], 'C': [np.nan] * 3})` do: (a) missing count per column; (b) `dropna` keeping only rows where `A` is present; (c) `dropna` removing columns that are **entirely** NaN; (d) count of non-missing values per column with `notna()`.

Expected (a) `A` 1, `B` 1, `C` 3; (b) rows 0 and 2; (c) columns `A`, `B`; (d) `A` 2, `B` 2, `C` 0.

In [14]:
df2 = pd.DataFrame({'A': [1, np.nan, 3], 'B': [4, 5, np.nan], 'C': [np.nan] * 3})
print(df2.isna().sum(axis=0))
print(df2.dropna(subset=['A']))
print(df2.dropna(axis=1, how='all'))
print(df2.notna().sum())

A    1
B    1
C    3
dtype: int64
     A    B   C
0  1.0  4.0 NaN
2  3.0  NaN NaN
     A    B
0  1.0  4.0
1  NaN  5.0
2  3.0  NaN
A    2
B    2
C    0
dtype: int64


### Exercise 3.4 
Fill the missing value of `s = pd.Series([10, np.nan, 30])` with the **median** of `s`.

Expected: `10.0, 20.0, 30.0`.

In [15]:
s = pd.Series([10, np.nan, 30])
print(s.fillna(s.median()))

0    10.0
1    20.0
2    30.0
dtype: float64


### Exercise 4.1 
For `df = pd.DataFrame({'ID': [101, 102, 101], 'Val': ['A', 'B', 'A']})` print the number of duplicated rows and then the table without duplicates based on `ID`.

Expected: `1`, then IDs 101 and 102.

In [16]:
df = pd.DataFrame({'ID': [101, 102, 101], 'Val': ['A', 'B', 'A']})
print(df.duplicated().sum())
print(df.drop_duplicates(subset=['ID']))

1
    ID Val
0  101   A
1  102   B


### Exercise 4.2 · Think
Now use `df = pd.DataFrame({'ID': [101, 102, 101], 'Val': ['A', 'B', 'C']})`. How many duplicates does `duplicated()` find on all columns, and how many with `subset=['ID']`? Which rows remain with `keep='first'`, `keep='last'` and `keep=False`? Predict first, then check. Which setting would you choose if the later record is the corrected one?

In [17]:
df = pd.DataFrame({'ID': [101, 102, 101], 'Val': ['A', 'B', 'C']})
print(df.duplicated().sum(), df.duplicated(subset=['ID']).sum())
for k in ['first', 'last', False]:
    print(k, df.drop_duplicates(subset=['ID'], keep=k).index.tolist())

0 1
first [0, 1]
last [1, 2]
False [1]


**Model answer:** All columns: 0 duplicates (the rows differ in `Val`). With `subset=['ID']`: 1. `keep='first'` keeps rows 0, 1; `keep='last'` keeps rows 1, 2; `keep=False` keeps only row 1 (every repeated ID is removed). If the later record is the corrected one, use `keep='last'`.

### Exercise 4.3 
Replace the impossible ages in `s = pd.Series([22, 150, 45, -5])` by NaN. Valid range: 0 to 100 (use `between` and `where`).

Expected: `22.0, NaN, 45.0, NaN`.

In [18]:
s = pd.Series([22, 150, 45, -5])
print(s.where(s.between(0, 100)))

0    22.0
1     NaN
2    45.0
3     NaN
dtype: float64


### Exercise 4.4 
Cap `pd.Series([5, 50, 120, 200])` between 10 and 100 using `clip`.

Expected: `10, 50, 100, 100`.

In [19]:
print(pd.Series([5, 50, 120, 200]).clip(lower=10, upper=100))

0     10
1     50
2    100
3    100
dtype: int64


### Exercise 5.1 
Clean `s = pd.Series(['  admin ', 'TECHNICAL', ' staff'])` so that it becomes `Admin`, `Technical`, `Staff` (chain `.str.strip()` and `.str.title()`).

In [20]:
s = pd.Series(['  admin ', 'TECHNICAL', ' staff'])
print(s.str.strip().str.title())

0        Admin
1    Technical
2        Staff
dtype: object


### Exercise 5.2 
Convert `pd.Series(['$1,200', '$3,500'])` to numbers: remove `$` and `,` with `str.replace(..., regex=False)`, then `pd.to_numeric`.

Expected: `1200, 3500` with `dtype: int64`.

In [21]:
p = pd.Series(['$1,200', '$3,500'])
print(pd.to_numeric(p.str.replace('$', '', regex=False).str.replace(',', '', regex=False)))

0    1200
1    3500
dtype: int64


### Exercise 5.3 
Split `pd.Series(['John Smith', 'Jane Doe'])` into a DataFrame with columns `First` and `Last` (`str.split` with `expand=True`).

In [22]:
n = pd.Series(['John Smith', 'Jane Doe'])
name_df = n.str.split(' ', n=1, expand=True)
name_df.columns = ['First', 'Last']
print(name_df)

  First   Last
0  John  Smith
1  Jane    Doe


### Exercise 6.1 
For `df = pd.DataFrame({'Zip': [10001, 90210]})` print the dtype of `Zip`, convert it to text with `astype`, and print the dtype again.

Expected: `int64`, then `object` (shown as `str` in pandas 3).

In [23]:
df = pd.DataFrame({'Zip': [10001, 90210]})
print(df['Zip'].dtype)
df['Zip'] = df['Zip'].astype(str)
print(df['Zip'].dtype)

int64
object


### Exercise 6.2 
Convert `s = pd.Series(['10', '20', 'INVALID', '40'])` with `to_numeric`: first with `errors='coerce'`.

Expected: `10.0, 20.0, NaN, 40.0`; the default raises `ValueError`.

In [24]:
s = pd.Series(['10', '20', 'INVALID', '40'])
print(pd.to_numeric(s, errors='coerce'))

0    10.0
1    20.0
2     NaN
3    40.0
dtype: float64


### Exercise 6.3 
Convert `pd.DataFrame({'Date': ['2023-01-01', 'bad']})['Date']` with `to_datetime`, `format='%Y-%m-%d'` and `errors='coerce'`.

Expected: `2023-01-01` and `NaT`.

In [25]:
d = pd.DataFrame({'Date': ['2023-01-01', 'bad']})
d['Date'] = pd.to_datetime(d['Date'], format='%Y-%m-%d', errors='coerce')
print(d)

        Date
0 2023-01-01
1        NaT


### Exercise 6.4 
Convert `['2023-01-01', '2023-05-15']` to dates and print the weekday name and the month number of each (`.dt`).

Expected: Sunday, Monday; months 1, 5.

In [26]:
dt = pd.to_datetime(pd.Series(['2023-01-01', '2023-05-15']))
print(dt.dt.day_name())
print(dt.dt.month)

0    Sunday
1    Monday
dtype: object
0    1
1    5
dtype: int32


### Exercise 7.1 
For `df = pd.DataFrame({'A': [1, 2], 'B': [3, 4], 'C': [5, 6]})` print the `type()` of `df['A']` and of `df[['A', 'B']]`.

Expected: `Series`, then `DataFrame`.

In [27]:
df = pd.DataFrame({'A': [1, 2], 'B': [3, 4], 'C': [5, 6]})
print(type(df['A']))
print(type(df[['A', 'B']]))

<class 'pandas.core.series.Series'>
<class 'pandas.core.frame.DataFrame'>


### Exercise 7.2 
For `df = pd.DataFrame({'Score': [88, 92]}, index=['Alice', 'Bob'])` get Alice's score with `loc` and again with `iloc`, then print Bob's whole row with `loc`.

Expected: `88`, `88`, then Bob's row with value 92.

In [28]:
df = pd.DataFrame({'Score': [88, 92]}, index=['Alice', 'Bob'])
print(df.loc['Alice', 'Score'])
print(df.iloc[0, 0])
print(df.loc['Bob'])

88
88
Score    92
Name: Bob, dtype: int64


### Exercise 7.3 
For `df = pd.DataFrame({'Age': [22, 35, 45], 'Dept': ['HR', 'IT', 'IT']})` show the rows where (a) Age > 30 **and** Dept is IT; (b) Age < 30 **or** Dept is IT; (c) **not** (a).

Expected: (a) rows 1, 2; (b) rows 0, 1, 2; (c) row 0.

In [29]:
df = pd.DataFrame({'Age': [22, 35, 45], 'Dept': ['HR', 'IT', 'IT']})
m = (df['Age'] > 30) & (df['Dept'] == 'IT')
print(df[m])
print(df[(df['Age'] < 30) | (df['Dept'] == 'IT')])
print(df[~m])

   Age Dept
1   35   IT
2   45   IT
   Age Dept
0   22   HR
1   35   IT
2   45   IT
   Age Dept
0   22   HR


### Exercise 7.4 
For `df = pd.DataFrame({'City': ['NY', 'London', 'Paris', 'Tokyo']})` select the rows whose city is in `['NY', 'Paris']`, then the rows whose city is **not** in that list.

In [30]:
df = pd.DataFrame({'City': ['NY', 'London', 'Paris', 'Tokyo']})
m = df['City'].isin(['NY', 'Paris'])
print(df[m])
print(df[~m])

    City
0     NY
2  Paris
     City
1  London
3   Tokyo


### Exercise 8.1 
For `df = pd.DataFrame({'Price': [100, 200], 'Qty': [2, 3]})` add `Subtotal` = Price × Qty and `Total_With_Tax` = Subtotal × 1.1 (no loops).

Expected: Subtotal 200, 600; Total_With_Tax 220.0, 660.0.

In [31]:
df = pd.DataFrame({'Price': [100, 200], 'Qty': [2, 3]})
df['Subtotal'] = df['Price'] * df['Qty']
df['Total_With_Tax'] = df['Subtotal'] * 1.1
print(df)

   Price  Qty  Subtotal  Total_With_Tax
0    100    2       200           220.0
1    200    3       600           660.0


### Exercise 8.2 
Use `pd.cut` on scores `[55, 78, 92, 40]` with bins `[0, 60, 80, 100]` and labels `Fail`, `Pass`, `Excellence`.

Expected: Fail, Pass, Excellence, Fail.

In [32]:
scores = pd.Series([55, 78, 92, 40])
print(pd.cut(scores, bins=[0, 60, 80, 100], labels=['Fail', 'Pass', 'Excellence']))

0          Fail
1          Pass
2    Excellence
3          Fail
dtype: category
Categories (3, object): ['Fail' < 'Pass' < 'Excellence']


### Exercise 8.3 
For `df = pd.DataFrame({'Dept': ['Sales', 'Sales', 'IT'], 'Salary': [50, 60, 80]})` compute the mean salary per department with `groupby`.

Expected: IT 80.0, Sales 55.0.

In [33]:
df = pd.DataFrame({'Dept': ['Sales', 'Sales', 'IT'], 'Salary': [50, 60, 80]})
print(df.groupby('Dept')['Salary'].mean())

Dept
IT       80.0
Sales    55.0
Name: Salary, dtype: float64


### Exercise 8.4 
For `df = pd.DataFrame({'Dept': ['Sales', 'Sales', 'IT'], 'Score': [80, 90, 95]})` use `groupby(...).agg(...)` with named outputs `Total_Staff` (count of `Score`) and `Avg_Score` (mean of `Score`).

Expected: IT 1 / 95.0; Sales 2 / 85.0.

In [34]:
df = pd.DataFrame({'Dept': ['Sales', 'Sales', 'IT'], 'Score': [80, 90, 95]})
print(df.groupby('Dept').agg(Total_Staff=('Score', 'count'), Avg_Score=('Score', 'mean')))

       Total_Staff  Avg_Score
Dept                         
IT               1       95.0
Sales            2       85.0


### Exercise 8.5 
With `emp_df = pd.DataFrame({'ID': [1, 2, 3], 'Name': ['Ann', 'Bob', 'Cy']})` and `sal_df = pd.DataFrame({'ID': [1, 2], 'Salary': [70, 80]})` run `pd.merge` with `how` = `'left'`, `'inner'`, `'right'`, `'outer'`. How many rows does each give?

Expected row counts: 3, 2, 2, 3.

In [35]:
emp_df = pd.DataFrame({'ID': [1, 2, 3], 'Name': ['Ann', 'Bob', 'Cy']})
sal_df = pd.DataFrame({'ID': [1, 2], 'Salary': [70, 80]})
for how in ['left', 'inner', 'right', 'outer']:
    r = pd.merge(emp_df, sal_df, on='ID', how=how)
    print(how, len(r))
print(pd.merge(emp_df, sal_df, on='ID', how='left'))

left 3
inner 2
right 2
outer 3
   ID Name  Salary
0   1  Ann    70.0
1   2  Bob    80.0
2   3   Cy     NaN


### Exercise 8.6 
With `df1 = pd.DataFrame({'A': [1, 2]})` and `df2 = pd.DataFrame({'A': [3, 4]})`: (a) stack them with `concat` and a fresh index (`ignore_index=True`); (b) with `df3 = pd.DataFrame({'B': [5, 6]})` place `df1` and `df3` side by side (`axis=1`).

Expected (a) `A` = 1, 2, 3, 4 with index 0 to 3; (b) columns `A`, `B`.

In [36]:
df1 = pd.DataFrame({'A': [1, 2]}); df2 = pd.DataFrame({'A': [3, 4]}); df3 = pd.DataFrame({'B': [5, 6]})
print(pd.concat([df1, df2], ignore_index=True))
print(pd.concat([df1, df3], axis=1))

   A
0  1
1  2
2  3
3  4
   A  B
0  1  5
1  2  6


---
## Part B — Think and solve with the HR data 

Now the real data. `raw` (from Exercise 2.1) is the messy table from slide 8, but larger (43 rows). Several questions ask you to **decide and explain**, not just to code. Write your reasoning in the answer cell below the code. Items marked ★ are optional.

### Exercise B1 · Think
Explore `raw` with `head()`, `info()`, `isna().sum()` and `unique()` on `dept`, `city`, `status`. List **at least 6 different data problems** in the answer cell. Also: the file contains the text `N/A` for employee 107's salary, but `raw` shows `NaN` there. Why? (Hint: `help(pd.read_csv)`, parameter `na_values`.)

In [37]:
print(raw.isna().sum())
for c in ['dept', 'city', 'status']:
    print(raw[c].unique())
print(raw.loc[raw['emp_id'] == 107, 'salary'])

emp_id        0
join_date     0
name          2
dept          0
age           8
salary        7
city          0
status        0
exit_date    29
dtype: int64
['Admin' 'TECHNICAL' 'staff' 'Technical' 'Staff' 'Admin ' ' technical '
 'ADMIN' 'Technical ' 'STAFF' ' admin ' 'Staff ']
['Delhi' 'delhi_' 'Mumbai' 'Pune' 'CHENNAI' 'Mumbai_' 'Kolkata' 'Chennai'
 'Kolkata_' 'MUMBAI' 'pune' 'PUNE' 'Chennai_' 'kolkata' 'DELHI']
['Active' 'Resigned' 'active' ' active ' 'Active ' ' resigned '
 'Resigned ' 'ACTIVE']
7    NaN
Name: salary, dtype: object


**Model answer:** Problems: (1) exact duplicate rows; (2) inconsistent case/spaces in `name`, `dept`, `status`; (3) stray `_` and case differences in `city`; (4) `join_date` is text in three formats; (5) `salary` is text with `$`, `,` and non-numbers like `TBD`; (6) impossible ages (250, -5, 199) and missing ages; (7) missing names and salaries; (8) `exit_date` is text. `read_csv` automatically converts strings such as `N/A`, `NA`, `NULL`, `nan` and empty fields to NaN (`na_values`), so 107's salary is already a real NaN, but `TBD` is not converted.

### Exercise B2 
Create `hr = raw.drop_duplicates().copy()`. Print the number of rows before and after, and check whether `emp_id` is now unique (`is_unique`).

Expected: 43 → 40 and `True`.

In [38]:
hr = raw.drop_duplicates().copy()
print(len(raw), '->', len(hr))
print(hr['emp_id'].is_unique)

43 -> 40
True


### Exercise B3 
Clean the text columns of `hr`: `name`, `dept`, `status` (strip, Title Case) and `city` (also remove the `_`). Then print the unique values of `dept`, `city`, `status`.

Expected: 3 departments, 5 cities, 2 statuses.

In [39]:
hr['name'] = hr['name'].str.strip().str.title()
hr['dept'] = hr['dept'].str.strip().str.title()
hr['status'] = hr['status'].str.strip().str.title()
hr['city'] = hr['city'].str.replace('_', '', regex=False).str.strip().str.title()
print(hr['dept'].unique(), hr['city'].unique(), hr['status'].unique())

['Admin' 'Technical' 'Staff'] ['Delhi' 'Mumbai' 'Pune' 'Chennai' 'Kolkata'] ['Active' 'Resigned']


### Exercise B4 
Clean the numbers in `hr`: make `salary` numeric (remove `$` and `,`; values that cannot be converted become NaN), make `age` numeric, and set every `age` outside 18–65 to NaN. Print the dtypes and `describe()` of both columns, and how many `age` values are missing now.

In [40]:
hr['salary'] = pd.to_numeric(hr['salary'].str.replace('$', '', regex=False).str.replace(',', '', regex=False), errors='coerce')
hr['age'] = pd.to_numeric(hr['age'], errors='coerce')
hr['age'] = hr['age'].where(hr['age'].between(18, 65))
print(hr[['salary', 'age']].dtypes)
print(hr[['salary', 'age']].describe())
print(hr['age'].isna().sum())

salary    float64
age       float64
dtype: object
             salary        age
count     25.000000  30.000000
mean   60800.000000  39.533333
std    11661.010534  10.064390
min    43500.000000  24.000000
25%    52000.000000  32.250000
50%    57500.000000  38.500000
75%    70500.000000  47.500000
max    80500.000000  57.000000
10


### Exercise B5 · Think
Convert `join_date` and `exit_date` of `hr` to real dates. `join_date` has **three formats** (`2024-01-15`, `15/02/2024`, `Mar 12, 2024`). Plan: parse with each format using `errors='coerce'`, then combine the results with `fillna`. Check that no `join_date` is `NaT`. Then explain: why is a date such as `03/04/2024` dangerous, and why does the file use `15/02/2024` rather than `03/04/2024`?

In [41]:
d1 = pd.to_datetime(hr['join_date'], format='%Y-%m-%d', errors='coerce')
d2 = pd.to_datetime(hr['join_date'], format='%d/%m/%Y', errors='coerce')
d3 = pd.to_datetime(hr['join_date'], format='%b %d, %Y', errors='coerce')
hr['join_date'] = d1.fillna(d2).fillna(d3)
hr['exit_date'] = pd.to_datetime(hr['exit_date'], format='%Y-%m-%d', errors='coerce')
print(hr['join_date'].isna().sum(), hr['exit_date'].isna().sum())

0 26


**Model answer:** `03/04/2024` can mean 3 April (day first) or March 4 (month first); a parser cannot know which, so a wrong guess silently gives a wrong date. `15/02/2024` is unambiguous because there is no month 15, so only the day-first format can match it.

### Exercise B6 · Think
`exit_date` is empty for employees who are still working. Compare the number of rows in `hr`, in `hr.dropna()` and in `hr.dropna(subset=['salary'])`. Which one is wrong for analysing salaries and why? Which employees disappear?

In [42]:
print(len(hr), len(hr.dropna()), len(hr.dropna(subset=['salary'])))

40 5 25


**Model answer:** `hr.dropna()` removes every row with *any* NaN, and `exit_date` is NaN for all active employees, so almost everyone disappears even though nothing is wrong with them. A missing `exit_date` is legitimate information. Use `subset=['salary']` so only the column you actually need decides which rows to drop.

### Exercise B7 · Think
Some salaries are missing. Compute the **mean salary per department** under three choices: (a) drop rows with missing salary; (b) fill with the overall median; (c) fill with the median of the employee's own department (hint: `hr.groupby('dept')['salary'].median()` and `hr['dept'].map(...)`). Also compare the standard deviation of `salary` after (a) and (b). Which choice would you use, and what is the effect of filling on the spread? Answer in the answer cell.

In [43]:
a = hr.dropna(subset=['salary']).groupby('dept')['salary'].mean()
sal_b = hr['salary'].fillna(hr['salary'].median())
b = sal_b.groupby(hr['dept']).mean()
dept_med = hr.groupby('dept')['salary'].median()
sal_c = hr['salary'].fillna(hr['dept'].map(dept_med))
c = sal_c.groupby(hr['dept']).mean()
print(pd.DataFrame({'drop': a, 'overall_median': b, 'dept_median': c}).round(0))
print(hr['salary'].std(), sal_b.std())

              drop  overall_median  dept_median
dept                                           
Admin      56312.0         56708.0      54958.0
Staff      47625.0         52562.0      47438.0
Technical  67615.0         64075.0      67400.0
11661.010533683033 9289.632696776856


**Model answer:** Salaries differ by department, so one overall median pulls every department towards the same value (it hides real differences). Filling with the department median is the most sensible of the three. Any filling lowers the standard deviation: the filled points add nothing to the sum of squares $\sum (x_i-\bar x)^2$ but the sample size in the denominator grows, so the data look less variable than they really are. Whatever you choose, report it.

### Exercise B8 
Answer the three questions from slide 8 using `hr`:
1. What is the average salary per department (also show the number of employees)? Use `groupby` + `agg`.
2. Which city (or cities, if there is a tie) has the most employees?
3. How many employees joined in each month? (Hint: `dt.month` and `value_counts()`, then `sort_index()`.)

In [44]:
print(hr.groupby('dept').agg(employees=('emp_id', 'count'), avg_salary=('salary', 'mean')).round(0))
print(hr['city'].value_counts())
print(hr['join_date'].dt.month.value_counts().sort_index())

           employees  avg_salary
dept                            
Admin             12     56312.0
Staff              8     47625.0
Technical         20     67615.0
city
Delhi      10
Mumbai     10
Pune        7
Kolkata     7
Chennai     6
Name: count, dtype: int64
join_date
1     2
2     1
3     6
4     3
5     1
6     3
7     4
8     2
9     8
10    5
11    4
12    1
Name: count, dtype: int64


### Exercise B9 · Think
The ages 250, -5 and 199 are impossible. Take the **original numeric** ages `pd.to_numeric(raw['age'])` and compare `age.where(age.between(18, 65))` with `age.clip(18, 65)`. What does each do to the value 250? Which is the right choice for an *age* column, and when would `clip` be reasonable?

In [45]:
age = pd.to_numeric(raw['age'])
print(age.where(age.between(18, 65)).describe())
print(age.clip(18, 65).describe())

count    31.000000
mean     39.290323
std       9.987304
min      24.000000
25%      32.000000
50%      36.000000
75%      47.000000
max      57.000000
Name: age, dtype: float64
count    35.000000
mean     40.885714
std      12.531775
min      18.000000
25%      32.000000
50%      41.000000
75%      50.500000
max      65.000000
Name: age, dtype: float64


**Model answer:** `where(...between...)` turns 250 and -5 into NaN: we admit we do not know the age. `clip(18, 65)` turns 250 into 65 and -5 into 18, which invents ages that were never recorded and biases the mean. For an impossible entry, NaN is the honest choice. `clip` is reasonable when the value is real but extreme and you want to limit its influence (for example capping a very large but genuine salary).

### Exercise B10 · Think
A new table `dept_budget` is given below. **Before running anything**, use `hr['dept'].value_counts()` to predict, in a comment, how many rows `pd.merge(hr, dept_budget, on='dept', how=...)` returns for `inner`, `left`, `right` and `outer`. Then run all four and compare. Which employees get `NaN` for `budget` in a left merge, and which budget row has no employees?

In [46]:
dept_budget = pd.DataFrame({'dept': ['Admin', 'Technical', 'Finance'], 'budget': [500, 900, 400]})
n = hr['dept'].value_counts()
print(n)
for how in ['inner', 'left', 'right', 'outer']:
    print(how, len(pd.merge(hr, dept_budget, on='dept', how=how)))
m = pd.merge(hr, dept_budget, on='dept', how='left')
print(m[m['budget'].isna()]['dept'].unique())

dept
Technical    20
Admin        12
Staff         8
Name: count, dtype: int64
inner 32
left 40
right 33
outer 41
['Staff']


**Model answer:** With $n_d$ employees in department $d$: inner = $n_{Admin}+n_{Technical}$; left = all employees (40); right = inner + 1 (the Finance row with no employees); outer = 40 + 1. Staff employees get NaN `budget` in the left merge; `Finance` has no employees.

### Exercise B11 · Think ★
Test the edge behaviour of `pd.cut`. Use `s = pd.Series([50000, 50001, 70000, np.nan])`, bins `[0, 50000, 70000, 200000]` and labels `Low`, `Mid`, `High`. Predict the label of each value with `right=True` (default) and with `right=False`, then check. What happens to the NaN?

In [47]:
s = pd.Series([50000, 50001, 70000, np.nan])
b, l = [0, 50000, 70000, 200000], ['Low', 'Mid', 'High']
print(pd.cut(s, bins=b, labels=l, right=True).tolist())
print(pd.cut(s, bins=b, labels=l, right=False).tolist())

['Low', 'Mid', 'Mid', nan]
['Mid', 'Mid', 'High', nan]


**Model answer:** `right=True` uses intervals $(a, b]$: 50000 → Low, 50001 → Mid, 70000 → Mid. `right=False` uses $[a, b)$: 50000 → Mid, 50001 → Mid, 70000 → High. NaN stays NaN (no bin).

### Exercise B12 · Think ★
Export `hr` to `hr_clean.csv` (`index=False`) and read it back. Print the dtypes of `join_date` and `exit_date` after reading. Why are they not dates any more? Fix it with the `parse_dates` parameter of `read_csv`.

In [48]:
hr.to_csv('hr_clean.csv', index=False)
print(pd.read_csv('hr_clean.csv').dtypes[['join_date', 'exit_date']])
print(pd.read_csv('hr_clean.csv', parse_dates=['join_date', 'exit_date']).dtypes[['join_date', 'exit_date']])

join_date    object
exit_date    object
dtype: object
join_date    datetime64[ns]
exit_date    datetime64[ns]
dtype: object


**Model answer:** A CSV file stores only text; dates are written as strings and the type information is lost. `read_csv` therefore returns `object` unless you ask for `parse_dates`. The same applies to category dtypes.

---
## Part C — Predict and debug 

**Predict:** write your prediction in the comment *before* you run the cell. **Debug:** the cell contains a mistake; run it, read the error or the strange output, then fix it.

### Exercise C1 · Predict
What does this print?

In [49]:
s = pd.Series([4, np.nan, 8])
print(s.mean(), s.count(), len(s))

6.0 2 3


**Model answer:** `6.0 2 3`: the mean skips NaN, so it is $(4+8)/2$, not $12/3$; `count` counts non-missing values, `len` counts all.

### Exercise C2 · Predict
Which rows remain?

In [50]:
df = pd.DataFrame({'A': [1, np.nan, 3], 'B': [np.nan, np.nan, 6]})
print(df.dropna(thresh=2))

     A    B
2  3.0  6.0


**Model answer:** Only row 2. `thresh=2` keeps rows with at least 2 non-missing values: row 0 has 1, row 1 has 0, row 2 has 2.

### Exercise C3 · Predict
How many distinct values in each line?

In [51]:
s = pd.Series(['  Admin ', 'ADMIN', 'admin'])
print(s.nunique())
print(s.str.strip().str.title().nunique())

3
1


**Model answer:** `3` and `1`. String comparison is exact, so the three spellings are different categories until they are standardised.

### Exercise C4 · Predict
What does this print?

In [52]:
print(pd.Series([10, 20, 30, 40]).clip(15, 35).tolist())

[15, 20, 30, 35]


**Model answer:** `[15, 20, 30, 35]`.

### Exercise C5 · Predict
The key `2` appears **twice** in each table. Predict the number of rows for `inner`, `left` and `outer`. (Think: how many pairs of matching rows exist for key 2?)

In [53]:
left = pd.DataFrame({'ID': [1, 2, 2], 'L': ['a', 'b', 'c']})
right = pd.DataFrame({'ID': [2, 2, 3], 'R': ['x', 'y', 'z']})
for how in ['inner', 'left', 'outer']:
    print(how, len(pd.merge(left, right, on='ID', how=how)))

inner 4
left 5
outer 6


**Model answer:** Inner 4: each of the 2 left rows with key 2 pairs with each of the 2 right rows (2 × 2). Left 5: those 4 plus the unmatched key 1. Outer 6: the 4 plus unmatched key 1 and unmatched key 3. In general the join has $\sum_k a_k b_k$ matched rows.

### Exercise C6 · Debug ★
We want rows with `Age` over 30 **and** `Dept` equal to `IT`. This code fails. Why? Fix it.

In [54]:
df = pd.DataFrame({'Age': [22, 35, 45], 'Dept': ['HR', 'IT', 'IT']})
print(df[(df['Age'] > 30) & (df['Dept'] == 'IT')])

   Age Dept
1   35   IT
2   45   IT


**Model answer:** `&` binds tighter than `>` and `==`, so Python first evaluates `30 & df['Dept']`. Each condition needs its own brackets.

### Exercise C7 · Debug ★
We expect a table with columns `First` and `Last`, but we get a Series of lists (and a warning). Fix it.

In [55]:
n = pd.Series(['John Smith', 'Jane Doe'])
name_df = n.str.split(' ', n=1, expand=True)
name_df.columns = ['First', 'Last']
print(name_df)

  First   Last
0  John  Smith
1  Jane    Doe


**Model answer:** With `expand=False` `split` returns a Series of lists, which has no columns. Use `expand=True` to get a DataFrame.

### Exercise C8 · Debug ★
We dropped the rows where `A` is missing, but printing `df` still shows the NaN. Why? Fix it.

In [56]:
df = pd.DataFrame({'A': [1, np.nan, 3], 'B': [4, 5, 6]})
df = df.dropna(subset=['A'])
print(df)

     A  B
0  1.0  4
2  3.0  6


**Model answer:** `dropna` returns a **new** DataFrame and leaves `df` unchanged. Assign the result back (`df = df.dropna(...)`).

### Exercise C9 · Debug ★
This conversion raises a `ValueError`. Why? Fix it so that it gives `1200` and `3500`.

In [57]:
p = pd.Series(['$1,200', '$3,500'])
print(pd.to_numeric(p.str.replace('$', '', regex=False).str.replace(',', '', regex=False)))

0    1200
1    3500
dtype: int64


**Model answer:** `$` and `,` are not part of a number, so `to_numeric` cannot parse the text. Remove the symbols first with `str.replace`.